# NISQA-SIM Mix-Only Generator

This notebook creates **40 mixed files** from `NISQA_TRAIN_SIM`, combining `REF` and `DEG` content
with randomized degradation regions.

Key behavior:
- Mix-only output (no REF-only / DEG-only files)
- Natural durations from source audio
- No zero-padding to a fixed length
- Randomized 1-3 degraded intervals per file

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display
from scipy.signal import resample_poly

In [ ]:
@dataclass
class Segment:
    """Represents a degradation interval in seconds."""

    start: float
    end: float


DATA_ROOT = Path("../data/raw/NISQA_Corpus")
SIM_SPLIT = "NISQA_TRAIN_SIM"
CSV_PATH = DATA_ROOT / SIM_SPLIT / f"{SIM_SPLIT}_file.csv"
OUTPUT_DIR = Path("../data/processed/nisqa_sim_mix_only_40")
MANIFEST_PATH = OUTPUT_DIR / "manifest.csv"

TOTAL_MIX_FILES = 40
TARGET_SAMPLE_RATE = 16000
MIN_DEG_SEGMENTS = 1
MAX_DEG_SEGMENTS = 3
MAX_DURATION_SECONDS = None
SEED = 42

DEGRADATION_COLUMNS = [
    "filter",
    "timeclipping",
    "wbgn",
    "p50mnru",
    "bgn",
    "clipping",
    "arb_filter",
    "asl_in",
    "asl_out",
    "codec1",
    "codec2",
    "codec3",
    "plcMode1",
    "plcMode2",
    "plcMode3",
]

rng = random.Random(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_audio_mono(path: Path) -> tuple[np.ndarray, int]:
    """Load waveform and convert to mono float32."""

    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    return audio.astype(np.float32), int(sr)


def resample_if_needed(audio: np.ndarray, sr_in: int, sr_out: int) -> np.ndarray:
    """Resample when the input sample rate differs from target rate."""

    if sr_in == sr_out:
        return audio
    gcd = math.gcd(sr_in, sr_out)
    up = sr_out // gcd
    down = sr_in // gcd
    return resample_poly(audio, up=up, down=down).astype(np.float32)


def align_pair(ref_audio: np.ndarray, deg_audio: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Align REF and DEG waveforms to a shared length without padding."""

    length = min(len(ref_audio), len(deg_audio))
    if MAX_DURATION_SECONDS is not None:
        max_len = int(round(MAX_DURATION_SECONDS * TARGET_SAMPLE_RATE))
        length = min(length, max_len)

    ref_audio = ref_audio[:length]
    deg_audio = deg_audio[:length]
    return ref_audio, deg_audio


def random_segments(total_seconds: float) -> list[Segment]:
    """Generate non-overlapping random degradation intervals."""

    if total_seconds <= 0.3:
        return [Segment(0.0, total_seconds)]

    max_possible = max(1, int(total_seconds // 0.35))
    n_segments = min(rng.randint(MIN_DEG_SEGMENTS, MAX_DEG_SEGMENTS), max_possible)

    min_len = max(0.18, total_seconds * 0.06)
    max_len = max(min_len, total_seconds * 0.55)

    accepted: list[Segment] = []
    attempts = 0
    while len(accepted) < n_segments and attempts < 400:
        attempts += 1
        seg_len = rng.uniform(min_len, max_len)
        seg_len = min(seg_len, total_seconds)
        start = rng.uniform(0.0, max(0.0, total_seconds - seg_len))
        candidate = Segment(start=start, end=start + seg_len)

        overlaps = any(
            not (candidate.end <= seg.start or candidate.start >= seg.end)
            for seg in accepted
        )
        if overlaps:
            continue

        accepted.append(candidate)

    if not accepted:
        fallback_len = min(total_seconds, max(0.2, total_seconds * 0.4))
        accepted = [Segment(0.0, fallback_len)]

    return sorted(accepted, key=lambda s: s.start)


def build_mix(ref_audio: np.ndarray, deg_audio: np.ndarray, sr: int) -> tuple[np.ndarray, list[Segment]]:
    """Create a mixed waveform by inserting DEG intervals into REF audio."""

    mixed = ref_audio.copy()
    total_seconds = len(mixed) / sr
    segments = random_segments(total_seconds=total_seconds)

    for seg in segments:
        i0 = max(0, min(int(round(seg.start * sr)), len(mixed)))
        i1 = max(i0, min(int(round(seg.end * sr)), len(mixed)))
        mixed[i0:i1] = deg_audio[i0:i1]

    return mixed, segments


def serialize_segments(segments: list[Segment]) -> str:
    """Serialize intervals to JSON for manifest storage."""

    payload = [{"start": round(s.start, 3), "end": round(s.end, 3)} for s in segments]
    return json.dumps(payload)


def timeline_from_deg_segments(total_seconds: float, deg_segments: list[Segment]) -> list[dict]:
    """Build an ordered REF/DEG timeline from degradation segments."""

    timeline: list[dict] = []
    cursor = 0.0

    for seg in sorted(deg_segments, key=lambda s: s.start):
        start = max(0.0, min(total_seconds, float(seg.start)))
        end = max(start, min(total_seconds, float(seg.end)))

        if start > cursor:
            timeline.append({"start": round(cursor, 3), "end": round(start, 3), "source": "REF"})

        if end > start:
            timeline.append({"start": round(start, 3), "end": round(end, 3), "source": "DEG"})
            cursor = end

    if cursor < total_seconds:
        timeline.append({"start": round(cursor, 3), "end": round(total_seconds, 3), "source": "REF"})

    return timeline


def switch_points_from_timeline(timeline: list[dict]) -> list[float]:
    """Return timeline boundaries where source may switch."""

    points: list[float] = []
    for item in timeline:
        points.extend([float(item["start"]), float(item["end"])])
    return sorted(set(round(p, 3) for p in points))


def is_active_tag(value: object) -> bool:
    """Return True when a metadata field indicates an active degradation."""

    if pd.isna(value):
        return False
    token = str(value).strip()
    return token not in {"", "-", "nan", "None"}


def extract_active_degradations(row: pd.Series) -> list[str]:
    """Extract active degradation tags from NISQA SIM metadata columns."""

    return [col for col in DEGRADATION_COLUMNS if is_active_tag(row.get(col, np.nan))]


In [ ]:
df = pd.read_csv(CSV_PATH)
df["ref_path"] = df["filepath_ref"].apply(lambda p: DATA_ROOT / p)
df["deg_path"] = df["filepath_deg"].apply(lambda p: DATA_ROOT / p)
df = df[df["ref_path"].apply(Path.exists) & df["deg_path"].apply(Path.exists)].copy()
df = df.reset_index(drop=True)

if len(df) < TOTAL_MIX_FILES:
    raise ValueError(f"Need at least {TOTAL_MIX_FILES} rows, found {len(df)}")

chosen_idx = rng.sample(list(df.index), TOTAL_MIX_FILES)
selected = df.loc[chosen_idx].reset_index(drop=True)
selected["active_degradation_types"] = selected.apply(extract_active_degradations, axis=1)
selected["active_degradation_types_json"] = selected["active_degradation_types"].apply(json.dumps)
selected["num_source_degradation_types"] = selected["active_degradation_types"].apply(len)

print(f"Selected {len(selected)} source pairs from {SIM_SPLIT}.")

tag_counts = selected["active_degradation_types"].explode().value_counts()
if len(tag_counts) > 0:
    display(tag_counts.rename("count").to_frame())
else:
    print("No explicit degradation tags were found in source metadata.")

display(
    selected[
        [
            "filename_deg",
            "active_degradation_types",
            "num_source_degradation_types",
        ]
    ].head(10)
)


In [ ]:
records: list[dict] = []

for idx, row in selected.iterrows():
    ref_audio, ref_sr = load_audio_mono(row["ref_path"])
    deg_audio, deg_sr = load_audio_mono(row["deg_path"])

    ref_audio = resample_if_needed(ref_audio, ref_sr, TARGET_SAMPLE_RATE)
    deg_audio = resample_if_needed(deg_audio, deg_sr, TARGET_SAMPLE_RATE)
    ref_audio, deg_audio = align_pair(ref_audio, deg_audio)

    mixed_audio, deg_segments = build_mix(ref_audio, deg_audio, TARGET_SAMPLE_RATE)

    stem = Path(row["filename_deg"]).stem
    out_path = OUTPUT_DIR / f"{idx:03d}_mix_{stem}.wav"
    sf.write(out_path, mixed_audio, TARGET_SAMPLE_RATE)

    duration_seconds = round(len(mixed_audio) / TARGET_SAMPLE_RATE, 3)
    text_segments = json.dumps([{"start": 0.0, "end": duration_seconds}])
    timeline = timeline_from_deg_segments(duration_seconds, deg_segments)
    switch_points = switch_points_from_timeline(timeline)

    records.append({
        "index": idx,
        "filename_ref": row["filename_ref"],
        "filename_deg": row["filename_deg"],
        "duration_seconds": duration_seconds,
        "text_segments": text_segments,
        "mix_deg_segments": serialize_segments(deg_segments),
        "switch_points": json.dumps(switch_points),
        "mix_timeline": json.dumps(timeline),
        "source_degradation_types": row["active_degradation_types_json"],
        "num_source_degradation_types": int(row["num_source_degradation_types"]),
        "mos": row.get("mos", np.nan),
    })

manifest_df = pd.DataFrame(records).sort_values("index").reset_index(drop=True)
manifest_df.to_csv(MANIFEST_PATH, index=False)

print(f"Wrote {len(manifest_df)} mixed files to {OUTPUT_DIR}")
print(f"Manifest: {MANIFEST_PATH}")
display(manifest_df[["duration_seconds", "num_source_degradation_types"]].describe())
display(manifest_df[["index", "filename_deg", "mix_deg_segments", "switch_points"]].head(10))


In [ ]:
preview = manifest_df.sample(n=min(6, len(manifest_df)), random_state=SEED)
for _, row in preview.iterrows():
    print()
    stem = Path(row["filename_deg"]).stem
    audio_path = OUTPUT_DIR / f"{int(row['index']):03d}_mix_{stem}.wav"
    print(f"mix_file={audio_path.name}")
    print(f"duration={row['duration_seconds']}s")
    print(f"degradation_types={row['source_degradation_types']}")
    print(f"switch_points={row['switch_points']}")
    print(f"mix_timeline={row['mix_timeline']}")
    display(Audio(filename=str(audio_path)))


## Output

- Audio files: `../data/processed/nisqa_sim_mix_only_40/*.wav`
- Manifest: `../data/processed/nisqa_sim_mix_only_40/manifest.csv`

Manifest schema:
- `index`
- `filename_ref`
- `filename_deg`
- `duration_seconds`
- `text_segments`
- `mix_deg_segments` (where DEG is inserted)
- `switch_points` (ordered boundary times for REF/DEG transitions)
- `mix_timeline` (ordered REF/DEG intervals)
- `source_degradation_types`
- `num_source_degradation_types`
- `mos`

To cap clip length later, set `MAX_DURATION_SECONDS` to a value like `8.0` or `10.0`.
